# Q//HORIZON 2026: Interactive Results

Run the cells below. Code is collapsed; click the bar on the left of a cell (or *View > Collapse*) to expand it. The same page is available outside Jupyter via `python dashboard/serve.py`.

In [ ]:
import subprocess, sys
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "dashboard").exists():
    ROOT = ROOT.parent
subprocess.run([sys.executable, str(ROOT / "dashboard" / "build_data.py")], check=True)

## 1. Interactive dashboard

In [ ]:
import html
from IPython.display import HTML, display
page = (ROOT / "dashboard" / "qhorizon_dashboard.html").read_text(encoding="utf-8")
display(HTML(f'<iframe srcdoc="{html.escape(page, quote=True)}" width="100%" height="900" style="border:1px solid #E2E8F0;border-radius:12px;background:#F8FAFC"></iframe>'))

## 2. Audit tables (10-seed averages, N = 500)

In [ ]:
import pandas as pd
from IPython.display import display
df = pd.read_csv(ROOT / "results" / "full" / "main_benchmark.csv")
df = df[df.n_train == 500]
T = df.groupby("model").agg(PR_AUC_mean=("test_pr_auc", "mean"), PR_AUC_std=("test_pr_auc", "std"), MCC=("test_mcc", "mean"), F1=("test_f1", "mean"), weights=("n_trainable_params", "first"), collapsed=("collapsed", "sum")).sort_values("PR_AUC_mean", ascending=False)
def tint(row):
    c = "#E9EEFC" if row.name.startswith(("LogReg", "RBF", "Random", "Tiny")) else ("#F3EBFC" if row.name.startswith(("Q1", "Q2")) else "#F1F5F9")
    return [f"background-color:{c}"] * len(row)
display(T.style.apply(tint, axis=1).format({"PR_AUC_mean": "{:.4f}", "PR_AUC_std": "{:.4f}", "MCC": "{:.3f}", "F1": "{:.3f}"}))

In [ ]:
import json
v = json.load(open(ROOT / "results" / "full" / "verdicts.json"))["hypotheses"]
V = pd.DataFrame([{"ID": k, "Comparison": h["comparison"], "Delta": round(h["mean_delta"], 4), "CI low": round(h["ci_lo"], 4), "CI high": round(h["ci_hi"], 4), "Verdict": h["verdict"]} for k, h in v.items()])
colors = {"Supported": "#DCFCE7", "Contradicted": "#FEE2E2", "Inconclusive": "#FEF3C7", "Practically tied": "#E0E7FF"}
display(V.style.map(lambda x: f"background-color:{colors.get(x, '')}", subset=["Verdict"]).hide(axis="index"))